# RAG：从文档摄取到可核验回答

这个 notebook 用纯 Python 搭建一个可执行的教学型 RAG。它不会调用外部向量数据库或大模型，而是把生产链路中最容易被隐藏的步骤——文档版本、ACL、切块、召回、融合、重排、上下文、引用和评估——全部显式展示。

## 学习目标

1. 画出离线索引与在线查询两条链路。
2. 理解 chunk、metadata、稀疏/稠密召回、rerank 和 context assembly 的职责边界。
3. 实现一个带 ACL 与版本过滤的 BM25 检索器。
4. 用 RRF 融合不同召回结果，并计算 Recall@k、MRR。
5. 区分召回失败、重排失败、上下文失败和生成不忠实。
6. 解释 RAG 为什么仍需防提示注入、知识库投毒和越权。

## 1. 生产级 RAG 的两条链路

离线：

```text
数据源 -> 解析/OCR -> 清洗去重 -> 结构化切块
      -> embedding/倒排/图索引 -> ACL、版本、来源 -> 索引发布
```

在线：

```text
鉴权 -> 查询理解/改写 -> 带过滤的多路召回 -> 融合去重
    -> rerank -> 父块扩展与上下文组装 -> 生成 -> 引用/忠实度校验
```

每一步都要保留 `request_id -> query -> candidate ids -> ranks -> final context -> answer` 的 trace。只有最终答案日志，无法判断错误发生在哪一层。

In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
from collections import Counter, defaultdict  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import re  # 导入本单元所需的依赖。
from typing import Iterable  # 导入本单元所需的依赖。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Document:  # 定义承载本节状态与行为的数据结构。
    doc_id: str  # 执行当前语句以推进本节示例。
    text: str  # 执行当前语句以推进本节示例。
    tenant: str  # 执行当前语句以推进本节示例。
    version: int  # 执行当前语句以推进本节示例。
    effective_date: str  # 执行当前语句以推进本节示例。
    source: str  # 执行当前语句以推进本节示例。

documents = [  # 计算并保存当前步骤的中间状态。
    Document("travel-v1", "旧版差旅制度：上海酒店上限为每晚500元。", "company-a", 1, "2024-01-01", "policy/travel-v1"),  # 执行当前语句以推进本节示例。
    Document("travel-v2", "新版差旅制度：上海酒店上限为每晚650元，北京为每晚600元。", "company-a", 2, "2026-01-01", "policy/travel-v2"),  # 执行当前语句以推进本节示例。
    Document("invoice", "报销必须提交电子发票，并在出差结束后30天内发起申请。", "company-a", 1, "2025-06-01", "policy/invoice"),  # 执行当前语句以推进本节示例。
    Document("secret-b", "B公司内部制度：酒店上限为每晚900元。", "company-b", 3, "2026-02-01", "private/tenant-b"),  # 执行当前语句以推进本节示例。
    Document("refund", "客户退款通常在审核通过后5个工作日原路返回。", "company-a", 1, "2025-12-01", "support/refund"),  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。

print(f"文档数: {len(documents)}")  # 执行当前语句以推进本节示例。
for doc in documents:  # 遍历输入元素以累积或检查结果。
    print(doc.doc_id, doc.tenant, doc.version, doc.text)  # 执行当前语句以推进本节示例。


## 2. 切块不是简单截字符串

chunk 应保留 `chunk_id、document_id、parent_id、标题路径、页码/offset、版本、时间、tenant、ACL、content_hash`。切得太小会丢上下文，太大则 embedding 稀释主题、召回噪声增多并浪费 prompt。overlap 能缓解边界断裂，却会造成重复召回和索引膨胀。

下面按句子构造简单 chunk。真实 PDF/表格应优先按标题、段落、表头和语义边界切分；字符窗口只是基线。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Chunk:  # 定义承载本节状态与行为的数据结构。
    chunk_id: str  # 执行当前语句以推进本节示例。
    doc_id: str  # 执行当前语句以推进本节示例。
    text: str  # 执行当前语句以推进本节示例。
    tenant: str  # 执行当前语句以推进本节示例。
    version: int  # 执行当前语句以推进本节示例。
    effective_date: str  # 执行当前语句以推进本节示例。
    source: str  # 执行当前语句以推进本节示例。

def sentence_chunks(docs: Iterable[Document]) -> list[Chunk]:  # 定义本节可复用的核心函数。
    chunks = []  # 计算并保存当前步骤的中间状态。
    for doc in docs:  # 遍历输入元素以累积或检查结果。
        sentences = [part.strip() for part in re.split(r"(?<=[。！？])", doc.text) if part.strip()]  # 计算并保存当前步骤的中间状态。
        for index, sentence in enumerate(sentences):  # 遍历输入元素以累积或检查结果。
            chunks.append(Chunk(  # 执行当前语句以推进本节示例。
                chunk_id=f"{doc.doc_id}#c{index}", doc_id=doc.doc_id, text=sentence,  # 计算并保存当前步骤的中间状态。
                tenant=doc.tenant, version=doc.version, effective_date=doc.effective_date, source=doc.source,  # 计算并保存当前步骤的中间状态。
            ))  # 执行当前语句以推进本节示例。
    return chunks  # 返回当前分支计算出的结果。

chunks = sentence_chunks(documents)  # 计算并保存当前步骤的中间状态。
for chunk in chunks:  # 遍历输入元素以累积或检查结果。
    print(chunk.chunk_id, "->", chunk.text)  # 执行当前语句以推进本节示例。


## 3. 稀疏检索：BM25

BM25 根据查询词在文档中的词频、逆文档频率和文档长度打分：

$$\operatorname{score}(q,d)=\sum_{t\in q}\operatorname{IDF}(t)\frac{f(t,d)(k_1+1)}{f(t,d)+k_1(1-b+b|d|/\operatorname{avgdl})}.$$

它擅长错误码、型号、姓名等精确词。中文生产系统应使用合适分词或学习型稀疏表示；下面用一个很小的领域词典做最长匹配，未知汉字退化为单字，只用于教学。

In [ ]:
LEXICON = sorted({  # 计算并保存当前步骤的中间状态。
    "差旅", "制度", "上海", "北京", "酒店", "上限", "每晚", "新版", "旧版",  # 执行当前语句以推进本节示例。
    "报销", "电子发票", "发票", "出差", "工作日", "退款", "审核", "申请", "公司",  # 执行当前语句以推进本节示例。
}, key=len, reverse=True)  # 计算并保存当前步骤的中间状态。

def tokenize_zh(text: str) -> list[str]:  # 定义本节可复用的核心函数。
    text = text.lower()  # 计算并保存当前步骤的中间状态。
    tokens, i = [], 0  # 计算并保存当前步骤的中间状态。
    while i < len(text):  # 在终止条件满足前持续推进状态。
        if text[i].isspace() or text[i] in "，。：；！？、.":  # 按当前条件选择后续控制路径。
            i += 1  # 计算并保存当前步骤的中间状态。
            continue  # 调整当前循环或占位控制流。
        ascii_match = re.match(r"[a-z0-9_-]+", text[i:])  # 计算并保存当前步骤的中间状态。
        if ascii_match:  # 按当前条件选择后续控制路径。
            token = ascii_match.group(0)  # 计算并保存当前步骤的中间状态。
            tokens.append(token)  # 执行当前语句以推进本节示例。
            i += len(token)  # 计算并保存当前步骤的中间状态。
            continue  # 调整当前循环或占位控制流。
        match = next((word for word in LEXICON if text.startswith(word, i)), None)  # 计算并保存当前步骤的中间状态。
        tokens.append(match or text[i])  # 执行当前语句以推进本节示例。
        i += len(match) if match else 1  # 计算并保存当前步骤的中间状态。
    return tokens  # 返回当前分支计算出的结果。

class BM25Index:  # 定义承载本节状态与行为的数据结构。
    def __init__(self, chunks: list[Chunk], k1: float = 1.5, b: float = 0.75):  # 定义本节可复用的核心函数。
        self.chunks, self.k1, self.b = chunks, k1, b  # 计算并保存当前步骤的中间状态。
        self.tokens = [tokenize_zh(chunk.text) for chunk in chunks]  # 计算并保存当前步骤的中间状态。
        self.term_freqs = [Counter(tokens) for tokens in self.tokens]  # 计算并保存当前步骤的中间状态。
        self.avgdl = sum(map(len, self.tokens)) / max(len(self.tokens), 1)  # 计算并保存当前步骤的中间状态。
        self.doc_freq = Counter()  # 计算并保存当前步骤的中间状态。
        for tokens in self.tokens:  # 遍历输入元素以累积或检查结果。
            self.doc_freq.update(set(tokens))  # 执行当前语句以推进本节示例。

    def score(self, query: str, index: int) -> float:  # 定义本节可复用的核心函数。
        score, n = 0.0, len(self.chunks)  # 计算并保存当前步骤的中间状态。
        frequencies = self.term_freqs[index]  # 计算并保存当前步骤的中间状态。
        doc_len = len(self.tokens[index])  # 计算并保存当前步骤的中间状态。
        for term in tokenize_zh(query):  # 遍历输入元素以累积或检查结果。
            df = self.doc_freq.get(term, 0)  # 计算并保存当前步骤的中间状态。
            idf = math.log(1 + (n - df + 0.5) / (df + 0.5))  # 计算并保存当前步骤的中间状态。
            tf = frequencies.get(term, 0)  # 计算并保存当前步骤的中间状态。
            denominator = tf + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)  # 计算并保存当前步骤的中间状态。
            if tf:  # 按当前条件选择后续控制路径。
                score += idf * tf * (self.k1 + 1) / denominator  # 计算并保存当前步骤的中间状态。
        return score  # 返回当前分支计算出的结果。

    def search(self, query: str, tenant: str, top_k: int = 3) -> list[tuple[Chunk, float]]:  # 定义本节可复用的核心函数。
        # ACL/tenant 条件在打分与 top-k 之前执行，避免无权结果占满候选。
        candidates = [  # 计算并保存当前步骤的中间状态。
            (chunk, self.score(query, index))  # 执行当前语句以推进本节示例。
            for index, chunk in enumerate(self.chunks)  # 遍历输入元素以累积或检查结果。
            if chunk.tenant == tenant  # 按当前条件选择后续控制路径。
        ]  # 执行当前语句以推进本节示例。
        return sorted(candidates, key=lambda item: (-item[1], item[0].chunk_id))[:top_k]  # 返回当前分支计算出的结果。

bm25 = BM25Index(chunks)  # 计算并保存当前步骤的中间状态。
query = "上海出差酒店报销上限是多少？"  # 计算并保存当前步骤的中间状态。
for chunk, score in bm25.search(query, tenant="company-a", top_k=4):  # 遍历输入元素以累积或检查结果。
    print(f"{score:.3f}	{chunk.chunk_id}	{chunk.text}")  # 执行当前语句以推进本节示例。


## 4. 稠密检索、混合召回与 RRF

真实稠密检索用双编码器产生向量 $u=f_q(q),v=f_d(d)$，以点积或余弦相似度搜索 ANN 索引。它擅长同义改写，却可能弱化精确编号、数字和否定。BM25 与 dense 的原始分数尺度不同，不能默认直接相加。

Reciprocal Rank Fusion 只使用名次：

$$\operatorname{RRF}(d)=\sum_r\frac{1}{c+\operatorname{rank}_r(d)}.$$

下面用“BM25 + 字符 bigram”模拟两路不同的词面召回，只用于解释融合；字符 bigram **不是**真正的语义 embedding。生产中可把第二路替换为向量检索结果。

In [ ]:
def char_bigrams(text: str) -> set[str]:  # 定义本节可复用的核心函数。
    clean = re.sub(r"\s+|[，。：；！？、.]", "", text.lower())  # 计算并保存当前步骤的中间状态。
    return {clean[i:i+2] for i in range(max(0, len(clean) - 1))}  # 返回当前分支计算出的结果。

def bigram_search(query: str, chunks: list[Chunk], tenant: str, top_k: int):  # 定义本节可复用的核心函数。
    q = char_bigrams(query)  # 计算并保存当前步骤的中间状态。
    scored = []  # 计算并保存当前步骤的中间状态。
    for chunk in chunks:  # 遍历输入元素以累积或检查结果。
        if chunk.tenant != tenant:  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        d = char_bigrams(chunk.text)  # 计算并保存当前步骤的中间状态。
        score = len(q & d) / len(q | d) if q | d else 0.0  # 计算并保存当前步骤的中间状态。
        scored.append((chunk, score))  # 执行当前语句以推进本节示例。
    return sorted(scored, key=lambda item: (-item[1], item[0].chunk_id))[:top_k]  # 返回当前分支计算出的结果。

def reciprocal_rank_fusion(rankings: list[list[tuple[Chunk, float]]], constant: int = 60):  # 定义本节可复用的核心函数。
    fused, by_id = defaultdict(float), {}  # 计算并保存当前步骤的中间状态。
    for ranking in rankings:  # 遍历输入元素以累积或检查结果。
        for rank, (chunk, _) in enumerate(ranking, start=1):  # 遍历输入元素以累积或检查结果。
            fused[chunk.chunk_id] += 1 / (constant + rank)  # 计算并保存当前步骤的中间状态。
            by_id[chunk.chunk_id] = chunk  # 计算并保存当前步骤的中间状态。
    return [(by_id[cid], score) for cid, score in sorted(fused.items(), key=lambda item: (-item[1], item[0]))]  # 返回当前分支计算出的结果。

lexical = bm25.search(query, tenant="company-a", top_k=4)  # 计算并保存当前步骤的中间状态。
bigrams = bigram_search(query, chunks, tenant="company-a", top_k=4)  # 计算并保存当前步骤的中间状态。
fused = reciprocal_rank_fusion([lexical, bigrams])  # 计算并保存当前步骤的中间状态。
print("BM25 rank:", [chunk.chunk_id for chunk, _ in lexical])  # 执行当前语句以推进本节示例。
print("bigram rank:", [chunk.chunk_id for chunk, _ in bigrams])  # 执行当前语句以推进本节示例。
print("RRF rank:   ", [(chunk.chunk_id, round(score, 4)) for chunk, score in fused])  # 执行当前语句以推进本节示例。


## 5. Rerank、版本规则与上下文组装

召回目标是用低成本获得高 Recall；reranker 对较小候选集做更精确的 query-document 联合判断。业务规则仍需显式处理：同一制度有多个版本时，纯相关性可能把旧版排在前面，必须结合生效时间、状态与来源优先级。

上下文组装还要去重、保留标题/表头、控制 token budget，并给每段分配不可伪造的引用 id。下面按 `doc_id` 前缀只保留最高版本，再在字符预算内构造证据。真实系统应按稳定的文档族 id 管理版本，而不是解析字符串。

In [ ]:
def document_family(doc_id: str) -> str:  # 定义本节可复用的核心函数。
    return re.sub(r"-v\d+$", "", doc_id)  # 返回当前分支计算出的结果。

def keep_latest(candidates: list[tuple[Chunk, float]]) -> list[tuple[Chunk, float]]:  # 定义本节可复用的核心函数。
    best = {}  # 计算并保存当前步骤的中间状态。
    for chunk, score in candidates:  # 遍历输入元素以累积或检查结果。
        family = document_family(chunk.doc_id)  # 计算并保存当前步骤的中间状态。
        current = best.get(family)  # 计算并保存当前步骤的中间状态。
        if current is None or chunk.version > current[0].version:  # 按当前条件选择后续控制路径。
            best[family] = (chunk, score)  # 计算并保存当前步骤的中间状态。
    return sorted(best.values(), key=lambda item: (-item[1], -item[0].version, item[0].chunk_id))  # 返回当前分支计算出的结果。

def assemble_context(candidates: list[tuple[Chunk, float]], char_budget: int = 120):  # 定义本节可复用的核心函数。
    selected, used = [], 0  # 计算并保存当前步骤的中间状态。
    for chunk, score in candidates:  # 遍历输入元素以累积或检查结果。
        block = f"[{chunk.chunk_id}] {chunk.text} 来源={chunk.source} 版本={chunk.version}"  # 计算并保存当前步骤的中间状态。
        if used + len(block) > char_budget:  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        selected.append((chunk, block))  # 执行当前语句以推进本节示例。
        used += len(block)  # 计算并保存当前步骤的中间状态。
    return selected  # 返回当前分支计算出的结果。

latest = keep_latest(fused)  # 计算并保存当前步骤的中间状态。
context = assemble_context(latest, char_budget=180)  # 计算并保存当前步骤的中间状态。
print("最终上下文：")  # 执行当前语句以推进本节示例。
for chunk, block in context:  # 遍历输入元素以累积或检查结果。
    print(block)  # 执行当前语句以推进本节示例。

# 教学型“生成器”：只展示基于证据作答和引用，不声称这是 LLM。
relevant = next((chunk for chunk, _ in context if "上海酒店上限" in chunk.text), None)  # 计算并保存当前步骤的中间状态。
answer = f"上海酒店上限为每晚650元 [{relevant.chunk_id}]" if relevant else "证据不足，无法回答。"  # 计算并保存当前步骤的中间状态。
print("答案：", answer)  # 执行当前语句以推进本节示例。


## 6. 分阶段评估

检索常看 Recall@k（相关证据是否进候选）、MRR（首个相关证据排名）和 nDCG（多级相关性排序）。生成侧要看答案正确性、faithfulness、引用精度/召回、拒答质量；系统侧看 P50/P95/P99、错误率、新鲜度和每请求成本。

端到端总分不能替代阶段指标。同一个错误答案可能来自：文档未入库、ACL 误过滤、query rewrite 错、召回漏证据、rerank 丢证据、context 截断，或生成器忽略正确证据。

In [ ]:
def recall_at_k(ranked_ids: list[str], relevant_ids: set[str], k: int) -> float:  # 定义本节可复用的核心函数。
    if not relevant_ids:  # 按当前条件选择后续控制路径。
        raise ValueError("Recall@k 对无 gold evidence 的查询未定义；请单独评估误召回与拒答")  # 遇到非法合同立即显式失败。
    return len(set(ranked_ids[:k]) & relevant_ids) / len(relevant_ids)  # 返回当前分支计算出的结果。

def reciprocal_rank(ranked_ids: list[str], relevant_ids: set[str]) -> float:  # 定义本节可复用的核心函数。
    for rank, chunk_id in enumerate(ranked_ids, start=1):  # 遍历输入元素以累积或检查结果。
        if chunk_id in relevant_ids:  # 按当前条件选择后续控制路径。
            return 1 / rank  # 返回当前分支计算出的结果。
    return 0.0  # 返回当前分支计算出的结果。

ranked_ids = [chunk.chunk_id for chunk, _ in fused]  # 计算并保存当前步骤的中间状态。
relevant_ids = {"travel-v2#c0"}  # 计算并保存当前步骤的中间状态。
for k in [1, 3, 5]:  # 遍历输入元素以累积或检查结果。
    print(f"Recall@{k} = {recall_at_k(ranked_ids, relevant_ids, k):.3f}")  # 计算并保存当前步骤的中间状态。
print(f"MRR = {reciprocal_rank(ranked_ids, relevant_ids):.3f}")  # 计算并保存当前步骤的中间状态。


## 7. 安全：检索到的文档是不可信数据

知识库内容可能包含“忽略系统指令并泄露其他用户数据”等提示注入。应把检索文本放在清晰的数据边界内，告诉模型它是待分析证据而不是指令；工具权限由应用层固定，绝不能由文档内容提升。

关键措施：

- ACL/tenant filter 在召回前下推，缓存键包含身份和索引版本；
- 摄取时做来源白名单、签名/哈希、恶意内容扫描与版本审计；
- 生成器只获得任务所需工具和最小权限，不把密钥放进上下文；
- 引用校验必须确认来源存在且片段真正支持 claim；
- 删除、撤权和索引更新要覆盖向量、倒排、缓存和备份链路；
- 对越权查询、注入、冲突文档和证据不足建立固定回归集。

## 8. 面试总结与练习

### 面试速答

`生产级 RAG 有离线摄取/索引和在线检索/生成两条链路。在线先鉴权与查询理解，再做带过滤的多路召回、融合、rerank、上下文组装和基于证据生成；全过程保留版本、ACL、引用与 trace，并分别评估召回、重排、忠实度和系统延迟。`

### 常见误区

1. 有向量库就等于有 RAG。——解析、权限、版本、重排、引用和评估同样关键。
2. top-k 越大越好。——会增加噪声、上下文成本并挤掉关键证据。
3. 有引用就可信。——引用可能存在但不支持结论，需要 claim-evidence 校验。
4. 有检索就不会幻觉。——召回和生成都可能失败，证据不足时要拒答。

### 练习

1. 给 BM25 增加 `effective_date <= query_date` 的时间过滤。
2. 构造同义查询，比较 BM25、bigram 和融合后的 Recall@k。
3. 增加一个恶意文档，验证它不能改变系统权限或工具调用。
4. 为每个请求输出 query、候选、rank、上下文和引用 trace。
5. 设计一张故障归因表，把 20 个坏例子分到 ingest、retrieve、rerank、context、generate 五层。